In [2]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

In [2]:
# from google.colab import drive
# drive.mount('/content/drive')

In [3]:
# !unzip /content/drive/MyDrive/DR-CNN/trainLabels_cropped.zip

In [3]:
import os

# Define the path to the directory containing the images
image_directory = '/kaggle/input/dr-dataset/resized_train_cropped/resized_train_cropped'

# Check if the directory exists
if os.path.exists(image_directory):
    # Get a list of all files and directories in the specified path
    all_items = os.listdir(image_directory)

    # Filter for image files (adjust extensions if needed)
    # Common image extensions: .jpg, .jpeg, .png, .gif, .bmp
    image_files = [item for item in all_items if item.lower().endswith(('.jpg', '.jpeg', '.png', '.gif', '.bmp'))]

    # Print the count of image files found
    print(f"Number of image files in '{image_directory}': {len(image_files)}")

else:
    print(f"Error: The directory '{image_directory}' does not exist.")

Number of image files in '/kaggle/input/dr-dataset/resized_train_cropped/resized_train_cropped': 35108


In [4]:
import pandas as pd
import os

df = pd.read_csv('/kaggle/input/dr-dataset/trainLabels_cropped.csv')

# Drop 'Unnamed: 0' if exists
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

# Keep only 'image' and 'level' columns
df = df[['image', 'level']]

# Add '.jpeg' to image filenames
df['image'] = df['image'].astype(str) + '.jpeg'

# Add full path to images
df['image'] = df['image'].apply(lambda x: os.path.join('/kaggle/input/dr-dataset/resized_train_cropped/resized_train_cropped', x))

print("\nLabel distribution:")
print(df['level'].value_counts())

print("\nSample with full image paths:")
print(df.head())



Label distribution:
level
0    25802
2     5288
1     2438
3      872
4      708
Name: count, dtype: int64

Sample with full image paths:
                                               image  level
0  /kaggle/input/dr-dataset/resized_train_cropped...      0
1  /kaggle/input/dr-dataset/resized_train_cropped...      0
2  /kaggle/input/dr-dataset/resized_train_cropped...      0
3  /kaggle/input/dr-dataset/resized_train_cropped...      0
4  /kaggle/input/dr-dataset/resized_train_cropped...      1


In [5]:
import os
import pathlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from torchvision import datasets, models, transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score
from torchvision.models import VGG16_Weights


In [6]:

# Create an empty list to hold samples
sampled_dfs = []

# Loop through each unique level
for level in sorted(df['level'].unique()):
    level_df = df[df['level'] == level]
    sampled = level_df.sample(2, random_state=42)  # fixed seed for reproducibility
    sampled_dfs.append(sampled)

# Combine all samples into one DataFrame
samples_per_level = pd.concat(sampled_dfs).reset_index(drop=True)

# Extract only the image file name from the full path
samples_per_level['image_name'] = samples_per_level['image'].apply(lambda x: os.path.basename(x))

# Display only level and image_name columns
print(samples_per_level[['level', 'image_name']])


   level        image_name
0      0   4095_right.jpeg
1      0   11527_left.jpeg
2      1    7797_left.jpeg
3      1  37962_right.jpeg
4      2    3525_left.jpeg
5      2   10043_left.jpeg
6      3  39521_right.jpeg
7      3   16727_left.jpeg
8      4    8462_left.jpeg
9      4  16007_right.jpeg


In [7]:
import os

# Check how many image paths in df['image'] actually exist on disk
df['exists'] = df['image'].apply(lambda x: os.path.exists(x))

# Summary
total = len(df)
exists_count = df['exists'].sum()
missing_count = total - exists_count

print(f"\nVerified: {exists_count} / {total} image paths exist.")
print(f"Missing: {missing_count} image paths do NOT exist.")

# If any are missing, show a few examples
if missing_count > 0:
    print("\nExamples of missing paths:")
    print(df[df['exists'] == False]['image'].head())



Verified: 35108 / 35108 image paths exist.
Missing: 0 image paths do NOT exist.


In [8]:
filtered_df = df[df['exists'] == True]
print (filtered_df)


                                                   image  level  exists
0      /kaggle/input/dr-dataset/resized_train_cropped...      0    True
1      /kaggle/input/dr-dataset/resized_train_cropped...      0    True
2      /kaggle/input/dr-dataset/resized_train_cropped...      0    True
3      /kaggle/input/dr-dataset/resized_train_cropped...      0    True
4      /kaggle/input/dr-dataset/resized_train_cropped...      1    True
...                                                  ...    ...     ...
35103  /kaggle/input/dr-dataset/resized_train_cropped...      0    True
35104  /kaggle/input/dr-dataset/resized_train_cropped...      0    True
35105  /kaggle/input/dr-dataset/resized_train_cropped...      0    True
35106  /kaggle/input/dr-dataset/resized_train_cropped...      0    True
35107  /kaggle/input/dr-dataset/resized_train_cropped...      1    True

[35108 rows x 3 columns]


In [9]:
from sklearn.model_selection import train_test_split

# Filter out only existing images
filtered_df = df[df['exists'] == True]

# Split into 80% training and 20% testing
train_df, test_df = train_test_split(
    filtered_df,
    test_size=0.2,
    stratify=filtered_df['level'],
    random_state=42
)

# Confirm sizes
print(f"Train set: {len(train_df)} images")
# print(f"Validation set: {len(val_df)} images")
print(f"Test set: {len(test_df)} images")


Train set: 28086 images
Test set: 7022 images


In [10]:
train_df['level'] = train_df['level'].astype(str)
# val_df['level'] = val_df['level'].astype(str)
test_df['level'] = test_df['level'].astype(str)

In [11]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Image size and batch size
IMG_SIZE = (224, 224)
BATCH_SIZE = 64

# Create a single generator for train/val/test
datagen = ImageDataGenerator(rescale=1./255)

train_gen = datagen.flow_from_dataframe(
    train_df,
    x_col='image',
    y_col='level',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

test_gen = datagen.flow_from_dataframe(
    test_df,
    x_col='image',
    y_col='level',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)


2025-06-04 12:52:24.479176: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1749041544.677921      35 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1749041544.739435      35 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Found 28086 validated image filenames belonging to 5 classes.
Found 7022 validated image filenames belonging to 5 classes.


In [12]:
##########################
### SETTINGS
##########################

RANDOM_SEED = 123
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


In [14]:
data_transforms = transforms.Compose([
    #resizes the input image to the specified size of (224, 224) pixels.
    transforms.Resize((224, 224)),
    #Convert Image to Pytorch tensor
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])

**Load the datasets with ImageFolder**

In [23]:
# ImageFolder: PyTorch utility for loading image data
train_data = train_df
test_data = test_df


**Create data loaders with batch size 16**

In [17]:
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)


In [13]:
batch_size = 16
# train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
# test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)

In [14]:
import torch
from PIL import Image

class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row['image']          # your image path column
        label_str = row['level']         # your label column as string

        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)

        label = int(label_str)           # Convert string label to int
        label = torch.tensor(label, dtype=torch.long)  # Convert to tensor

        return image, label


In [15]:
from torch.utils.data import WeightedRandomSampler

# Get labels as a list of integers
train_labels = train_df['level'].astype(int).tolist()

# Compute class weights
from sklearn.utils.class_weight import compute_class_weight
class_weights = compute_class_weight('balanced', classes=np.unique(train_labels), y=train_labels)

# Map weights to each sample
sample_weights = [class_weights[label] for label in train_labels]

In [16]:
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)


In [27]:
from torchvision import transforms

#Training transform with augmentation
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

#Testing transform (no augmentation)
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

train_dataset = CustomDataset(train_df, transform=train_transform)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, shuffle=True)

test_dataset = CustomDataset(test_df, transform=test_transform)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=False)


In [18]:
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)


cuda


In [19]:
print(train_df['level'].unique())
print(test_df['level'].unique())


['0' '3' '1' '2' '4']
['2' '0' '3' '4' '1']


In [20]:
for inputs, labels in train_loader:
    print("Batch labels unique:", labels.unique())
    break  # just one batch for debug


Batch labels unique: tensor([0, 1, 2, 3, 4])


In [21]:
print(labels.dtype)  # should print torch.int64 (alias torch.long)


torch.int64


In [29]:
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

# Load the pre-trained EfficientNet-B0 model
model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)

# Modify the classifier for 5 classes
num_classes = 5
model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

# Training loop
num_epochs = 5
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)
    print(f'Epoch {epoch}/{num_epochs - 1}, Loss: {epoch_loss:.4f}')


Epoch 0/4, Loss: 0.7252
Epoch 1/4, Loss: 0.6209
Epoch 2/4, Loss: 0.5752
Epoch 3/4, Loss: 0.5507
Epoch 4/4, Loss: 0.5150


**Testing**

In [30]:
from sklearn.metrics import classification_report

# Testing phase
model.eval()
all_preds = []
all_labels = []
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Accuracy
accuracy = accuracy_score(all_labels, all_preds)
print(f'Testing Accuracy: {accuracy:.4f}')

# Classification Report
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=['0', '1', '2', '3', '4']))


Testing Accuracy: 0.8189

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.96      0.91      5161
           1       0.50      0.01      0.02       488
           2       0.64      0.59      0.61      1058
           3       0.53      0.47      0.49       174
           4       0.72      0.56      0.63       141

    accuracy                           0.82      7022
   macro avg       0.65      0.52      0.53      7022
weighted avg       0.79      0.82      0.79      7022

